# 0. Foundation

Builds the frozen files every section uses. The Producer runs this **once**; everyone else only uses the outputs:

- `clean_songs.csv`: one row per unique song, with a `split` column (train/test) and a `fold` column (0-4)
- `predictions/test_predictions.csv`: the final model's predictions on the test set

Frozen on Nov 20. Don't re-run or edit after that, so everyone's numbers stay identical.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from common import FEATURES, load, final_model

df = pd.read_csv("dataset.csv")

## Data cleaning: group sub-genres into 5 parent genres

In [2]:
broad_genre_map = {
    
    # Pop-ish music
    "pop": "pop",
    "power-pop": "pop",
    "synth-pop": "pop",
    "indie-pop": "pop",
    "k-pop": "pop",
    "j-pop": "pop",
    "pop-film": "pop",
    "children": "pop",
    "kids": "pop",
    "j-dance": "pop",
    "j-idol": "pop",
    "mandopop": "pop",
    "cantopop": "pop",

    # Rock and metal
    "rock": "rock",
    "alt-rock": "rock",
    "alternative": "other", #rock was my worst performing on everything so i went back and killed the more genre ambiguous subgenres
    "indie": "other",       #same as above
    "punk": "rock",
    "punk-rock": "rock",
    "psych-rock": "rock",
    "hard-rock": "rock",
    "rock-n-roll": "rock",
    "rockabilly": "rock",
    "grunge": "rock",
    "metal": "rock",
    "heavy-metal": "rock",
    "black-metal": "rock",
    "death-metal": "rock",
    "metalcore": "rock",
    "goth": "rock",
    "emo": "rock",
    "grindcore": "rock",

    # Hip-hop / R&B
    "hip-hop": "hiphop/rnb",
    "r-n-b": "hiphop/rnb",
    "soul": "hiphop/rnb",
    "funk": "hiphop/rnb",
    "groove": "other",
    

    # Electronic
    "edm": "electronic",
    "electronic": "electronic",
    "electro": "electronic",
    "house": "electronic",
    "deep-house": "electronic",
    "progressive-house": "electronic",
    "chicago-house": "electronic",
    "minimal-techno": "electronic",
    "techno": "electronic",
    "trance": "electronic",
    "dubstep": "electronic",
    "drum-and-bass": "electronic",
    "breakbeat": "electronic",
    "idm": "electronic",
    "club": "electronic",
    "dance": "electronic",
    "dancehall": "electronic",
    "garage": "electronic",
    "trip-hop": "electronic",

    # Latin
    "latin": "latin",
    "latino": "latin",
    "salsa": "latin",
    "samba": "latin",
    "pagode": "latin",
    "sertanejo": "latin",
    "mpb": "latin",
    "brazil": "latin",
    "forro": "latin",
    "reggaeton": "latin",
    "reggae": "latin",
    "tango": "latin",

    # Jazz/ Blues
    "jazz": "jazz-blues",
    "blues": "jazz-blues",
    
    # Country
    "bluegrass": "country",
    "folk": "country",
    "country": "country",
    "honky-tonk": "country",

    # Classical
    "classical": "classical",
    "opera": "classical",
    "piano": "classical",
    "ambient": "classical",
    "new-age": "classical",

    # other
    "afrobeat": "other",
    "indian": "other",
    "iranian": "other",
    "turkish": "other",
    "malay": "other",
    "world-music": "other",
    "french": "other",
    "german": "other",
    "spanish": "other",
    "swedish": "other",
    "acoustic": "other",
    "chill": "other",
    "happy": "other",
    "sad": "other",
    "sleep": "other",
    "study": "other",
    "party": "other",
    "romance": "other",
    "gospel": "other",
    "comedy": "other",
    "disco": "other",
    "disney": "other",
    "show-tunes": "other",
    "guitar": "other",
    "songwriter": "other",
    "singer-songwriter": "other"
}

df["broad_genre"] = df["track_genre"].map(broad_genre_map)
major_genres = ["pop", "rock", "hiphop/rnb", "electronic", "latin"]
df = df[df["broad_genre"].isin(major_genres)]

## Remove duplicate songs and define failure

The same song often appears under several genre tags or as a re-release. Keep one copy of each.

In [3]:
df = df.drop_duplicates("track_id").drop_duplicates(["artists", "track_name"])
df["failure"] = (df["popularity"] < 30).astype(int)

## Split by artist

20% of artists go to the test set, so no artist is on both sides. The training rows get 5 cross-validation folds, also grouped by artist.

In [4]:
artists = df["artists"].fillna("unknown")
_, test_rows = next(GroupShuffleSplit(test_size=0.2, random_state=42).split(df, groups=artists))
df["split"] = "train"
df.iloc[test_rows, df.columns.get_loc("split")] = "test"

train = df["split"] == "train"
df["fold"] = -1
for k, (_, val_rows) in enumerate(GroupKFold(5).split(df[train], groups=artists[train])):
    df.loc[df.index[train][val_rows], "fold"] = k

keep = ["track_id", "artists", "track_name", "track_genre", "broad_genre", "popularity", "failure"] + FEATURES + ["split", "fold"]
df[keep].to_csv("clean_songs.csv", index=False)
print(len(df), "unique songs |", train.sum(), "train |", (~train).sum(), "test")
df.groupby("split")["failure"].mean().round(3)

47077 unique songs | 37371 train | 9706 test


split
test     0.376
train    0.398
Name: failure, dtype: float64

## Final model predictions on the test set

In [5]:
d = load()
model = final_model().fit(d.X_train, d.y_train)

test = d.songs[d.songs["split"] == "test"][["track_id", "artists", "track_name", "broad_genre", "failure"]].copy()
test["proba"] = model.predict_proba(d.X_test)[:, 1]
Path("predictions").mkdir(exist_ok=True)
test.to_csv("predictions/test_predictions.csv", index=False)
test.head()

,track_id,artists,track_name,broad_genre,failure,proba
25,4nzIpIeHWiuFQnOV5ZELnj,Deftones,Ohms,rock,1,0.640473
75,6VuztO5yaKP4VjqSkZxpan,Ashes Remain,On My Own,rock,0,0.300095
128,0L7lTy1aRn1TYYCTazcO5j,The Seige,Run for Your Life,rock,0,0.242870
130,42T2QQv3xgBlpQxaSP7lnK,Creed,One Last Breath,rock,0,0.318019
132,7K0WZ75nHqj450OYd3ZywF,The EverLove,I'm Dangerous,rock,0,0.356385
